# 제2장 제3절 Matplotlib과 Seaborn 시각화

2027 개정 원고의 실습 자료입니다. 위에서 아래로 실행합니다. 데이터 파일 준비와 버전은 [실행 안내](https://github.com/Lilcob/AICE_Associate/blob/main/docs/getting_started.md)를 확인하세요.

건강검진 CSV와 iris.data를 노트북의 현재 폴더에 준비합니다. 실제 건강검진 자료는 2023년 자료이며 개정판 연도와 구분합니다.

설치나 버전 변경이 필요하면 안내에 따라 준비한 뒤 런타임을 재시작하세요. 저장된 출력은 비워 두었으며 직접 실행한 값을 확인합니다.


In [ ]:
import numpy as np
import pandas as pd
print('NumPy', np.__version__, '| pandas', pd.__version__)


In [ ]:
from pathlib import Path
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_paths = [Path('/System/Library/Fonts/Supplemental/AppleGothic.ttf'),
              Path('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')]
font_path = next((p for p in font_paths if p.is_file()), None)
if font_path is None:
    try:
        import google.colab
    except ImportError:
        raise RuntimeError('한글 글꼴을 설치하거나 font_paths에 글꼴 경로를 지정하세요.')
    import subprocess
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-nanum'], check=True)
    font_path = Path('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
font_manager.fontManager.addfont(str(font_path))
_aice_font_name = font_manager.FontProperties(fname=str(font_path)).get_name()
plt.rcParams['font.family'] = _aice_font_name
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# 핵심 개념: 데이터에 숨겨진 이야기 발견하기

In [ ]:
import pandas as pd
df = pd.read_csv("국민건강보험공단_건강검진정보_2023.CSV", encoding="cp949")

df_shape = df.shape
df_type = df.dtypes
df_missing_values = df.isnull().sum()
df_head = df.head()
print(df_shape)
print(df_type)
print(df_missing_values)
print(df_head)

## 상관 분석

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
corr_matrix = df[['체중(5kg단위)', '허리둘레']].corr()  # ①
print(corr_matrix.round(2))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')  # ②
plt.title('상관 행렬')
plt.show()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)  # ③
plt.title('상관 행렬 (vmin/vmax 적용)', fontsize=14)
plt.show()
cross_tab = pd.crosstab(df['성별코드'], df['흡연상태'])  # ④
print(cross_tab)
cross_tab_margins = pd.crosstab(df['성별코드'],df['흡연상태'],margins=True)
print(cross_tab_margins)
cross_tab_norm_index = pd.crosstab(df['성별코드'],df['흡연상태'],normalize='index')  # ⑤
print(cross_tab_norm_index.round(3))
cross_tab_norm_all = pd.crosstab(df['성별코드'],df['흡연상태'],normalize='all'
)
print(cross_tab_norm_all.round(3))
sns.heatmap(cross_tab, annot=True, fmt='d', cmap='Blues')  # ⑥
plt.title('성별 vs 흡연상태 교차표(Heatmap)')
plt.xlabel('흡연상태')
plt.ylabel('성별코드')
plt.show()
cross_tab.plot(kind='bar', stacked=True, figsize=(8, 6))  # ⑦
plt.title('성별 vs 흡연상태(Stacked Bar)')
plt.xlabel('성별코드')
plt.ylabel('빈도')
plt.legend(title='흡연상태', loc='upper left')
plt.show()

### 데이터 유형과 시각화 선택

In [ ]:
sns.countplot(data=df, x='성별코드')     # ①
plt.title('성별 분포')                   # ②
plt.xlabel('성별코드 (1: 남성, 2: 여성)')  # ③
plt.ylabel('빈도수')                    # ④
plt.show()                             # ⑤

In [ ]:
sns.lineplot(data=df, x='연령대코드(5세단위)', y='허리둘레', hue='성별코드') # ①
plt.title('연령대별 남녀 허리둘레 추이')  # ②
plt.xlabel('연령대 (5세 단위)')        # ③
plt.ylabel('허리둘레 (cm)')           # ④
plt.show()                          # ⑤

### 데이터 분석에서 시각화의 활용

####선 그래프 (Line Plot) - 추세 확인

In [ ]:
df_grouped = df.groupby(['연령대코드(5세단위)', '흡연상태'])['체중(5kg단위)'].mean().reset_index()  # ①
age_unique = sorted(df_grouped['연령대코드(5세단위)'].unique())
waist_1 = [
    df_grouped[
        (df_grouped['연령대코드(5세단위)'] == age) &
        (df_grouped['흡연상태'] == 1.0)
    ]['체중(5kg단위)'].values[0]
    if len(df_grouped[
        (df_grouped['연령대코드(5세단위)'] == age) &
        (df_grouped['흡연상태'] == 1.0)
    ]) > 0 else np.nan
    for age in age_unique
]  # ②
waist_2 = [
    df_grouped[
        (df_grouped['연령대코드(5세단위)'] == age) &
        (df_grouped['흡연상태'] == 2.0)
    ]['체중(5kg단위)'].values[0]
    if len(df_grouped[
        (df_grouped['연령대코드(5세단위)'] == age) &
        (df_grouped['흡연상태'] == 2.0)
    ]) > 0 else np.nan
    for age in age_unique
]
waist_3 = [
    df_grouped[
        (df_grouped['연령대코드(5세단위)'] == age) &
        (df_grouped['흡연상태'] == 3.0)
    ]['체중(5kg단위)'].values[0]
    if len(df_grouped[
        (df_grouped['연령대코드(5세단위)'] == age) &
        (df_grouped['흡연상태'] == 3.0)
    ]) > 0 else np.nan
    for age in age_unique
]
plt.figure(figsize=(10, 6))  # ③
plt.plot(age_unique, waist_1,
         color='blue', linewidth=2.5,
         marker='o', markersize=6,
         linestyle='-',
         label='비흡연 (1.0)')
plt.plot(age_unique, waist_2,
         color='orange', linewidth=2.5,
         marker='s', markersize=6,
         linestyle='--',
         label='과거흡연 (2.0)')
plt.plot(age_unique, waist_3,
         color='red', linewidth=2.5,
         marker='^', markersize=6,
         linestyle=':',
         label='현재흡연 (3.0)')  # ④
plt.xlim(1, 19)
plt.ylim(50, 80)
plt.xticks(range(1, 20, 2))
plt.xlabel('연령대 (5세 단위)', fontweight='bold')
plt.ylabel('체중 (5kg 단위)', fontweight='bold')
plt.title(
    '연령대별 흡연 상태에 따른 체중 변화 (Matplotlib 선 그래프)',
    fontsize=14
)  # ⑤
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3, axis='both')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6)) # ①
sns.lineplot(
data=df,x='연령대코드(5세단위)',y='체중(5kg단위)',hue='흡연상태',style='흡연상태',markers=True,dashes=False,errorbar=('ci', 95),palette='Set1') # ②
plt.title('연령대별 흡연 상태에 따른 체중 변화 (Seaborn 선 그래프)',fontsize=14)
plt.xlabel('연령대 (5세 단위)', fontweight='bold')
plt.ylabel('체중 (5kg 단위)', fontweight='bold')
plt.xticks(rotation=0) # ③
plt.legend(title='흡연상태', loc='upper left')
sns.set_style('whitegrid', rc={'font.family': _aice_font_name}) # ④
plt.tight_layout()
plt.show() 

####산점도 그래프 (Scatter Plot) 와 회귀선 (Reg Plot) -  두 변수 관계와 선형 회귀 추정

In [ ]:
import numpy as np
from scipy.stats import linregress
df_male = df[df['성별코드']==1][['체중(5kg단위)','허리둘레']]
df_male = df_male.dropna().head(1000)
df_female = df[df['성별코드']==2][['체중(5kg단위)','허리둘레']]
df_female = df_female.dropna().head(1000)
plt.figure(figsize=(7, 5))  # ①
plt.scatter(df_male['체중(5kg단위)'], df_male['허리둘레'], s=30, alpha=0.6,
            marker='o', color='blue', edgecolors='black', linewidth=0.5, label='남성')
plt.scatter(df_female['체중(5kg단위)'], df_female['허리둘레'], s=30, alpha=0.6,marker='s', color='red', edgecolors='black', linewidth=0.5, label='여성')  # ②
slope_m, intercept_m, r, p, std = linregress(df_male['체중(5kg단위)'], df_male['허리둘레'])
line_m = slope_m * df_male['체중(5kg단위)'] + intercept_m
plt.plot(df_male['체중(5kg단위)'], line_m, 'b--', linewidth=2, label=f'남성 회귀선 (r={r:.2f})')  # ③
slope_f, intercept_f, r_f, p_f, std_f = linregress(df_female['체중(5kg단위)'], df_female['허리둘레'])
line_f = slope_f * df_female['체중(5kg단위)'] + intercept_f
plt.plot(df_female['체중(5kg단위)'], line_f, 'r--', linewidth=2, label=f'여성 회귀선 (r={r_f:.2f})')
plt.xlabel('체중 (5kg 단위)', fontweight='bold')
plt.ylabel('허리둘레 (cm)', fontweight='bold')
plt.title('체중 vs. 허리둘레 (Matplotlib 산점도)', fontsize=14)  # ④
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
df_sample = df[['체중(5kg단위)', '허리둘레', '성별코드']].dropna()
df_sample = df_sample.head(1000)
plt.figure(figsize=(7, 5))  # ①
sns.scatterplot(data=df_sample,x='체중(5kg단위)',y='허리둘레',hue='성별코드',size='성별코드',sizes=(20, 100),alpha=0.6,palette='Set1')  # ②
sns.regplot(data=df_sample,x='체중(5kg단위)',y='허리둘레',scatter=False,color='gray',line_kws={'linestyle':'--', 'linewidth':2})  # ③
plt.title('체중 vs. 허리둘레 (Seaborn 산점도 + Reg Plot)', fontsize=14)
plt.xlabel('체중 (5kg 단위)', fontweight='bold')
plt.ylabel('허리둘레 (cm)', fontweight='bold')
plt.legend(title='성별')  # ④
sns.set_style('darkgrid', rc={'font.family': _aice_font_name})  # ⑤
plt.tight_layout()
plt.show()

In [ ]:
df_sample = df[['체중(5kg단위)', '허리둘레', '성별코드']].dropna()
df_sample = df_sample.head(1000)
plt.figure(figsize=(7, 5))  # ①
sns.scatterplot(data=df_sample,x='체중(5kg단위)',y='허리둘레',hue='성별코드',size='성별코드',sizes=(20, 100),alpha=0.6,palette='Set1')  # ②
sns.regplot(data=df_sample,x='체중(5kg단위)',y='허리둘레',scatter=False,color='gray',line_kws={'linestyle':'--', 'linewidth':2})  # ③
plt.title('체중 vs. 허리둘레 (Seaborn 산점도 + Reg Plot)', fontsize=14)
plt.xlabel('체중 (5kg 단위)', fontweight='bold')
plt.ylabel('허리둘레 (cm)', fontweight='bold')
plt.legend(title='성별')  # ④
sns.set_style('darkgrid', rc={'font.family': _aice_font_name})  # ⑤
plt.tight_layout()
plt.show()

In [ ]:
df_sample = df[['허리둘레', 'HDL콜레스테롤', '성별코드']].dropna().head(1000)  # ①
g = sns.lmplot(data=df_sample,x='허리둘레',y='HDL콜레스테롤',hue='성별코드', lowess=True,height=5, aspect=1.4,palette='Set1', scatter_kws={'alpha':0.6, 's':30},line_kws={'linestyle':'--', 'linewidth':2})  # ②
g.set_axis_labels('허리둘레 (cm)', 'HDL콜레스테롤 (mg/dL)')
g.set(title='허리둘레 vs. HDL (LOWESS, 성별별)')  # ③
sns.set_style('darkgrid', rc={'font.family': _aice_font_name})
plt.tight_layout()
plt.show()

####박스플롯  (Boxplot), 막대 그래프 (Bar Plot) - 카테고리별 비교


In [ ]:
data_male = [
   df[(df['성별코드']==1) & (df['흡연상태']==1.0)]['허리둘레'].dropna().head(500).values,
   df[(df['성별코드']==1) & (df['흡연상태']==2.0)]['허리둘레'].dropna().head(500).values,
   df[(df['성별코드']==1) & (df['흡연상태']==3.0)]['허리둘레'].dropna().head(500).values
]
data_female = [
   df[(df['성별코드']==2) & (df['흡연상태']==1.0)]['허리둘레'].dropna().head(500).values,
   df[(df['성별코드']==2) & (df['흡연상태']==2.0)]['허리둘레'].dropna().head(500).values,
   df[(df['성별코드']==2) & (df['흡연상태']==3.0)]['허리둘레'].dropna().head(500).values
]
all_data = data_male + data_female
plt.figure(figsize=(8, 5))  # ①
box = plt.boxplot(all_data,notch=True,showfliers=True,positions=[1,2,3,5,6,7],tick_labels=['남-비', '남-과', '남-현', '여-비', '여-과', '여-현'],patch_artist=True,whiskerprops=dict(color='gray'))  # ②
colors = ['blue', 'blue', 'blue', 'red', 'red', 'red']
for patch, color in zip(box['boxes'], colors):  # ③
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
plt.xlabel('성별-흡연상태', fontweight='bold')
plt.ylabel('허리둘레 (cm)', fontweight='bold')
plt.title('성별 흡연별 허리둘레 (Matplotlib 박스 플롯)', fontsize=14)  # ④
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
df_sample = df.dropna(subset=['성별코드', '흡연상태', '허리둘레']).head(500)
plt.figure(figsize=(8, 5))  # ①
sns.boxplot(data=df_sample,x='흡연상태',y='허리둘레', hue='성별코드',showfliers=True,width=0.6,palette='Set3')  # ②
plt.title('성별 흡연별 허리둘레 (Seaborn 박스 플롯)', fontsize=14)
plt.xlabel('흡연상태', fontweight='bold')
plt.ylabel('허리둘레 (cm)', fontweight='bold')
plt.xticks(rotation=0)
sns.set_style('white', rc={'font.family': _aice_font_name})  # ③
plt.tight_layout()
plt.show()

In [ ]:
mean_waist = df.groupby('흡연상태')['허리둘레'].mean()
std_waist = df.groupby('흡연상태')['허리둘레'].std()
plt.figure(figsize=(6, 4))  # ①
bars = plt.bar(mean_waist.index,mean_waist.values,width=0.6,color=['lightgreen', 'yellow', 'red'],edgecolor='black',linewidth=1,yerr=std_waist.values,capsize=5,
alpha=0.8)  # ②
labels = ['비흡연 평균', '과거 평균', '현재 평균']
for i, bar in enumerate(bars):  # ③
   bar.set_label(labels[i])
plt.xlim(0.5, 4.5)
plt.xticks([1, 2, 3], ['비흡연', '과거', '현재'])
plt.xlabel('흡연상태', fontweight='bold')
plt.ylabel('허리둘레 평균 (cm)', fontweight='bold')
plt.title('흡연 상태별 허리둘레 (Matplotlib 막대 그래프)', fontsize=14)  # ④
plt.legend(loc='upper right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))  # ①
sns.barplot(data=df, x='흡연상태', y='허리둘레', errorbar='sd',
            capsize=0.1, hue='흡연상태', legend=False, 
            palette='viridis')  # ②
plt.title('흡연 상태별 허리둘레 (Seaborn 막대 그래프)', fontsize=14)  # ③
plt.xlabel('흡연상태 (1: 비흡연, 2: 과거, 3: 현재)', fontweight='bold')  # ④
plt.ylabel('허리둘레 평균 (cm)', fontweight='bold')  # ⑤
plt.xticks(rotation=0)  # ⑥
sns.set_style('white', rc={'font.family': _aice_font_name})  # ⑦
plt.tight_layout()  # ⑧
plt.show()  # ⑨

####히스토그램 (Histogram), 파이그래프 (Pie Chart) - 분포 확인


In [ ]:
year_code = df['연령대코드(5세단위)'].dropna()  # ①
plt.figure(figsize=(8, 5))  # ②
plt.hist(year_code, bins=50, density=False, histtype='bar', color='orange',
        edgecolor='black', alpha=0.7, label='연령대코드(5세단위) 분포')  # ③
plt.xlabel('연령대코드(5세단위)', fontweight='bold')  # ⑤
plt.ylabel('빈도', fontweight='bold')  # ⑥
plt.title('연령대코드(5세단위) 분포 (Matplotlib 히스토그램)', fontsize=14)  # ⑦
plt.legend(loc='upper right')  # ⑧
plt.grid(axis='y', alpha=0.3)  # ⑨
plt.tight_layout()  # ⑩
plt.show()  # ⑪

In [ ]:
plt.figure(figsize=(8, 5))  # ①
sns.histplot(data=df, x='연령대코드(5세단위)', bins=30, kde=True,
             hue='성별코드', multiple='dodge', palette='pastel')  # ②
plt.title('연령대코드(5세단위) 분포 (Seaborn 히스토그램)', fontsize=14)  # ③
plt.xlabel('연령대코드(5세단위)', fontweight='bold')  # ④
plt.ylabel('빈도', fontweight='bold')  # ⑤
sns.set_style('whitegrid', rc={'font.family': _aice_font_name})  # ⑥
plt.tight_layout()  # ⑦
plt.show()  # ⑧

In [ ]:
smoking_counts = df['흡연상태'].value_counts().reindex([1.0, 2.0, 3.0], fill_value=0)
labels = ['비흡연', '과거 흡연', '현재 흡연']
plt.figure(figsize=(6, 6))  # ①
plt.pie(smoking_counts.values, labels=labels, autopct='%1.1f%%',
        startangle=90, colors=['lightgreen', 'yellow', 'red'],
        explode=(0.1, 0, 0), shadow=True)  # ②
plt.title('흡연상태 비중 (Matplotlib 파이 그래프)', fontsize=14)  # ③
plt.show()  # ④

In [ ]:
plt.figure(figsize=(6, 6)) # ①
plt.pie(smoking_counts.values, labels=labels, autopct='%1.1f%%',
        startangle=90, explode=(0.1, 0, 0), shadow=True) # ②
plt.title('흡연상태 비중 (Seaborn 스타일 파이 그래프)', fontsize=14) # ③
plt.show()  # ④

####고급 비교/관계 시각화(Pairplot, JointPlot, Swamplot)


In [ ]:
df_sample = df[['체중(5kg단위)', '허리둘레', 'HDL콜레스테롤', '성별코드']].dropna().head(1000)  # ①
sns.pairplot(df_sample, hue='성별코드', diag_kind='kde',
             plot_kws={'alpha': 0.6}, palette='Set1')  # ②
plt.suptitle('건강 변수 Pairplot (Seaborn)', fontsize=16, y=1.02)  # ③
plt.show()  # ④

In [ ]:
df_sample = df[['체중(5kg단위)', '허리둘레', '성별코드']].dropna().head(1000)  # ①
g = sns.jointplot(data=df_sample, x='체중(5kg단위)', y='허리둘레',
          hue='성별코드', kind='scatter', alpha=0.6, palette='Set1') # ②
g.fig.subplots_adjust(top=0.90)  # ③
g.fig.suptitle('체중 vs. 허리둘레 Jointplot (Seaborn)', fontsize=14, y=0.98)  # ③
plt.show()  # ④

In [ ]:
df_sample = df[['흡연상태', '허리둘레', '성별코드']].dropna().head(1000) # ①
sns.swarmplot(data=df_sample, x='흡연상태', y='허리둘레', hue='성별코드',
              palette='Set2', size=3, alpha=0.7)  # ②
plt.title('흡연 상태별 허리둘레 Swarmplot (Seaborn)', fontsize=14)  
plt.xlabel('흡연상태 (1:비, 2:과, 3:현)', fontweight='bold')  
plt.ylabel('허리둘레 (cm)', fontweight='bold')  
plt.xticks(rotation=0)  # ③
plt.tight_layout() 
plt.show()  

#### 서브 플롯(Subplot) 으로 다양한 그래프 한번에 확인하기

In [ ]:
df_sample = df[['체중(5kg단위)', '허리둘레', 'HDL콜레스테롤', 
                '흡연상태', '성별코드']].dropna().head(1000)  # ①
fig, axes = plt.subplots(1, 3, figsize=(15, 5))  # ②
axes = axes.ravel()  
sns.scatterplot(data=df_sample, x='체중(5kg단위)', y='HDL콜레스테롤',
              hue='성별코드', alpha=0.6, palette='Set1', ax=axes[0])# ③
axes[0].set_title('체중 vs. HDL (Scatter)') 
sns.scatterplot(data=df_sample, x='체중(5kg단위)', y='허리둘레',
              hue='성별코드',alpha=0.6, palette='Set1', ax=axes[1])  
sns.regplot(data=df_sample, x='체중(5kg단위)', y='허리둘레', scatter=False,
           color='gray', line_kws={'linestyle':'--'}, ax=axes[1])   # ④
axes[1].set_title('체중 vs. 허리둘레 (w/ Reg Line)')  
sns.swarmplot(data=df_sample, x='흡연상태', y='허리둘레', hue='성별코드',
             palette='Set1', size=3, alpha=0.7, ax=axes[2])
axes[2].set_title('흡연 상태별 허리둘레 (Swarm)') 
axes[2].set_xlabel('흡연상태 (1:비, 2:과, 3:현)') 
plt.suptitle('건강 지표 통합 비교: Subplot (Seaborn + Matplotlib)',
             fontsize=16, y=1.02)   # ⑥
sns.set_style('darkgrid', rc={'font.family': _aice_font_name})  # ⑥ 
plt.tight_layout()  # ⑥ 
plt.show()  

# 단계별 실습: 건강검진 데이터셋 시각화하기

##건강검진 데이터셋 직관적 분석하기


건강검진 데이터셋 컬럼 설명(데이터 파일명(csv)): 국민건강보험공단_건강검진정보_2023.CSV
연령대·성별·지역처럼 숫자로 표시한 코드의 의미는 해당 파일의 코드북에서 확인합니다. 연령대코드를 실제 나이로 읽거나 다른 연도 파일의 코드 정의를 그대로 적용하지 않도록 주의하세요.
성별코드 : 성별 (1: 남성, 2: 여성)
신장(5cm단위) : 신장 (5cm 단위)
체중(5kg단위) : 체중 (5kg 단위)
허리둘레 : 허리 둘레 (cm)
수축기혈압 : 수축기 혈압 (mmHg)
식전혈당(공복혈당) : 공복 혈당 (mg/dL)
총콜레스테롤 : 총 콜레스테롤 (mg/dL)
흡연상태 : 흡연 상태 (1: 비흡연, 2: 과거흡연, 3: 현재흡연)

### 데이터 불러오기

연습 문제1: 데이터 분석을 위해 Pandas와 Seaborn, Matplotlib 라이브러리를 불러오세요.  
&nbsp;&nbsp;&nbsp;&nbsp;① Pandas를 pd로 임포트하세요.  
&nbsp;&nbsp;&nbsp;&nbsp;② Seaborn을 sns로 임포트하세요.  
&nbsp;&nbsp;&nbsp;&nbsp;③ Matplotlib.pyplot을 plt로 임포트하세요.


In [ ]:
import pandas as pd # ①
import seaborn as sns # ②
import matplotlib.pyplot as plt # ③

연습 문제 2: 건강검진 데이터 파일을 읽어 데이터프레임 변수에 저장하세요.  
&nbsp;&nbsp;&nbsp;&nbsp;① 국민건강보험공단_건강검진정보_2023.CSV 파일을 pd.read_csv()로 읽어 df_sample 변수에 할당하세요.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; - **컬럼**: ["연령대코드(5세단위)", "성별코드", "신장(5cm단위)",  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;"체중(5kg단위)", "허리둘레", "수축기혈압",  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;"식전혈당(공복혈당)", "총콜레스테롤", "흡연상태"]  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; - **인코딩**: cp949  
&nbsp;&nbsp;&nbsp;&nbsp;② 데이터의 기본 정보를 확인하는 코드를 추가하세요 (info()와 head() 사용).  


In [ ]:
import numpy as np
def remove_outliers_iqr_multi(df, columns=None, q_low=0.25, q_high=0.75):
    """IQR 기반 이상치 제거(또는 필터링).
    기본은 Q1=25%, Q3=75% 사용. q_low/q_high 조절 가능."""
    if columns is None:
        columns = df.select_dtypes(include=[np.number]).columns.tolist()
    df_filtered = df.copy()
    removed_count = 0
    for col in columns:
        Q1 = df_filtered[col].quantile(q_low)
        Q3 = df_filtered[col].quantile(q_high)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        initial_len = len(df_filtered)

        mask = df_filtered[col].isna() | ((df_filtered[col] >= lower) & (df_filtered[col] <= upper))

        df_filtered = df_filtered.loc[mask]

        removed = initial_len - len(df_filtered) # 24
        removed_count += removed
        print(f"Column '{col}': Removed {removed} outliers (Bounds: [{lower:.2f}, {upper:.2f}])")

    print(f"Total removed: {removed_count}")
    return df_filtered

In [ ]:
df_sample = pd.read_csv('국민건강보험공단_건강검진정보_2023.CSV',
                usecols=["연령대코드(5세단위)", "성별코드", "신장(5cm단위)",
                         "체중(5kg단위)", "허리둘레", "수축기혈압",
                         "식전혈당(공복혈당)", "총콜레스테롤", "흡연상태"],
                encoding="cp949").dropna() # ①
continuous_cols = ['신장(5cm단위)', '체중(5kg단위)', '허리둘레',
                   '수축기혈압', '식전혈당(공복혈당)', '총콜레스테롤']
df_sample = remove_outliers_iqr_multi(df_sample, continuous_cols)
df_sample.info() # ②
print(df_sample.head()) # ②

### 시각화로 데이터 분석하기

추가 연습: 연령대코드별 평균 수축기혈압을 선 그래프로 확인하세요. 연령대코드는 실제 나이가 아닌 5세 단위의 코드입니다.

In [ ]:
sns.lineplot(data=df_sample, x='연령대코드(5세단위)', y='수축기혈압', errorbar=('ci', 95)) # ①
plt.title('연령대별 평균 수축기혈압 추이') # ②
plt.xlabel('연령대 (5세 단위)') # ②
plt.ylabel('평균 수축기혈압 (mmHg)') # ②
plt.show() # ①

연습 문제 3: 산점도 그래프(Scatter Plot)와 회귀선(Regression Line)을 활용하여 체중과 수축기혈압의 관계를 성별에 따라 탐색하세요.  
① 체중(5kg단위)을 x축, 수축기혈압을 y축으로 하는 산점도를 그리세요. hue='성별코드'로 색상 구분하세요.  
② 각 그룹에 회귀선을 추가하세요 (lmplot 활용, lowess=False로 선형 회귀).  
③ 그래프 제목을 '체중 vs. 수축기혈압 (성별 구분)'으로 설정하세요.  


In [ ]:
sns.lmplot(data=df_sample, x='체중(5kg단위)', y='수축기혈압', hue='성별코드', 
           lowess=False)
plt.title('체중 vs. 수축기혈압 (성별 구분)')
plt.show()

연습 문제 4: 박스플롯(Box Plot)과 막대그래프(Bar Plot)를 활용하여 연령대별 식전혈당 분포와 흡연상태별 평균 혈당을 비교하세요.  
① 연령대별 식전혈당의 박스플롯을 그리세요 (sns.boxplot, x='연령대코드(5세단위)', y='식전혈당(공복혈당)').  
② 흡연상태별 평균 식전혈당 막대그래프를 그리세요 (sns.barplot, x='흡연상태', y='식전혈당(공복혈당)').  
③ 두 그래프를 subplot으로 나란히 배치하세요 (figsize=(12,5)).  


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=df_sample, 
            x='연령대코드(5세단위)', y='식전혈당(공복혈당)', ax=ax1)
ax1.set_title('연령대별 식전혈당 분포 (박스플롯)')
sns.barplot(data=df_sample, x='흡연상태', y='식전혈당(공복혈당)', ax=ax2)
ax2.set_title('흡연상태별 평균 식전혈당 (막대그래프)')
plt.tight_layout()
plt.show()

연습 문제 5: 히스토그램(Histogram)과 파이차트(Pie Chart)를 활용하여 총콜레스테롤 분포와 성별 비율을 시각화하세요.  
① 총콜레스테롤의 히스토그램을 그리세요 (sns.histplot, bins=30, kde=True).  
② 성별코드의 비율을 파이차트로 그리세요 (value_counts() 후 plt.pie).  
③ 두 그래프를 subplot으로 배치하세요 (figsize=(10,5)).  


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
sns.histplot(data=df_sample, x='총콜레스테롤', bins=30, kde=True, ax=ax1)
ax1.set_title('총콜레스테롤 분포 (히스토그램)')
gender_counts = df_sample['성별코드'].value_counts()
ax2.pie(gender_counts.values, labels=gender_counts.index,
        autopct='%1.1f%%')
ax2.set_title('성별 비율 (파이차트)')
plt.tight_layout()
plt.show()

연습 문제 6: Pairplot, Jointplot, Swarmplot을 활용하여 다변량 관계를 탐색하세요.  
① Pairplot으로 신장, 체중, 허리둘레 간 쌍대 쌍(pairwise) 관계를 그리세요 (sns.pairplot, hue='성별코드').  
② 체중과 BMI(체중/신장^2로 계산) 간 Jointplot을 그리세요 (sns.jointplot, kind='hex' 또는 'kde').  

In [ ]:
df_sample_finalplot = df_sample.head(500).copy()
sns.pairplot(df_sample_finalplot[['신장(5cm단위)', '체중(5kg단위)', 
                                  '허리둘레', '성별코드']], 
                                 hue='성별코드', diag_kind='hist')
plt.suptitle('신장-체중-허리둘레 Pairplot (성별 구분)')
plt.show()
df_sample_finalplot['BMI'] = df_sample_finalplot['체중(5kg단위)'] / (df_sample_finalplot['신장(5cm단위)']/100)**2
sns.jointplot(data=df_sample_finalplot, x='체중(5kg단위)', y='BMI',
              kind='kde')
plt.suptitle('체중 vs. BMI Jointplot (KDE)')
plt.show()
sns.swarmplot(data=df_sample_finalplot,
              x='흡연상태', y='연령대코드(5세단위)', hue='성별코드')  # ③
plt.title('연령대-흡연상태 Swarmplot (성별 구분)')
plt.show()

## 개념도와 추가 설명

[핵심 개념 노트북](https://colab.research.google.com/github/Lilcob/AICE_Associate/blob/main/notebooks/2027_concepts.ipynb)에서 개념도의 작은 예시를 실행할 수 있습니다.
